# Gaitkeeper v2: Upgraded Adversarial Patch

**Loss:** Multi-objective (Uncertainty + IoU + Confidence + Edge)  
**Method:** EoT-PGD with full real-world transform suite  
**Optimizer:** Adam (replaces raw sign updates)

> Set Runtime to T4 GPU before running.


---

## Cell 1: Install


In [ ]:
!pip install ultralytics opencv-python-headless matplotlib scipy -q

---

## Cell 2: Imports & Config


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms.functional as TF
import random
import warnings

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# CONFIG — tune these values as you experiment
# ============================================================

YOLO_MODEL = "yolov8n-seg"  # swap to yolov8x-seg for final eval
PERSON_CLASS = 0  # COCO person class ID

# Patch location on person box
# Shirt region: horizontal center, from 15% to 55% down the person box
PATCH_SCALE = 0.4  # fraction of person box width covered by patch
SHIRT_TOP = 0.15  # how far down the box the shirt starts
SHIRT_BOT = 0.55  # how far down the box the shirt ends

# Optimization
EPSILON = 0.1  # max total pixel change allowed (0=none, 1=full)
LR = 0.01  # Adam learning rate
NUM_EPOCHS = 50  # training epochs
EOT_N = 10  # transforms to average per step

# Multi-objective loss weights (must sum to 1.0)
W_UNCERTAINTY = 0.4
W_IOU = 0.3
W_CONFIDENCE = 0.2
W_EDGE = 0.1

# EoT transform ranges
ROT_RANGE = (-20, 20)  # degrees
SCALE_RANGE = (0.85, 1.15)  # multiplicative
BRIGHTNESS_RANGE = (0.7, 1.3)
CONTRAST_RANGE = (0.8, 1.2)
NOISE_STD = 0.05
BLUR_MAX = 3  # max kernel size for motion blur
PERSPECTIVE_PROB = 0.5  # probability of applying perspective warp

print("Config loaded.")

---

## Cell 3: Load Model


In [ ]:
model = YOLO(YOLO_MODEL)
model.to(DEVICE)
torch_model = model.model
torch_model.eval()
print(f"Loaded {YOLO_MODEL}")

---

## Cell 4: Load Image


In [ ]:
import urllib.request

# --- Option A: upload from your machine ---
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = list(uploaded.keys())[0]

# --- Option B: use a local file already in Colab ---
# IMAGE_PATH = '/content/your_frame.jpg'

# --- Option C: placeholder (has people in it) ---
urllib.request.urlretrieve(
    "https://ultralytics.com/images/bus.jpg", "/content/test.jpg"
)
IMAGE_PATH = "/content/test.jpg"

orig_bgr = cv2.imread(IMAGE_PATH)
orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 6))
plt.imshow(orig_rgb)
plt.title("Input Image")
plt.axis("off")
plt.show()
print(f"Image shape: {orig_rgb.shape}")

---

## Cell 5: Baseline Inference


In [ ]:
def run_inference(img_rgb, title="YOLOv8-seg"):
    results = model(img_rgb, verbose=False)
    result = results[0]
    ann = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

    confs = (
        [
            float(result.boxes.conf[i])
            for i, c in enumerate(result.boxes.cls)
            if int(c) == PERSON_CLASS
        ]
        if result.boxes is not None
        else []
    )

    avg_conf = float(np.mean(confs)) if confs else 0.0
    has_masks = result.masks is not None

    plt.figure(figsize=(10, 7))
    plt.imshow(ann)
    plt.title(
        f"{title}  |  persons={len(confs)}  conf={avg_conf:.3f}  "
        f"{'masks OK' if has_masks else 'NO MASKS'}"
    )
    plt.axis("off")
    plt.show()

    print(f"[{title}] persons={len(confs)}, avg_conf={avg_conf:.4f}, masks={has_masks}")
    return results, avg_conf


baseline_results, baseline_conf = run_inference(orig_rgb, "Baseline (Clean)")
print(f"\nTarget to beat: {baseline_conf:.4f}")

---

## Cell 6: Helpers — Image I/O, Shirt Box, Patch Compositing


In [ ]:
def img_to_tensor(img_rgb):
    """HxWx3 uint8 numpy -> 1x3xHxW float32 on DEVICE"""
    t = torch.from_numpy(img_rgb).float() / 255.0
    return t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)


def tensor_to_img(t):
    """1x3xHxW float tensor -> HxWx3 uint8 numpy"""
    t = t.squeeze(0).permute(1, 2, 0).detach().cpu()
    return (t.clamp(0, 1) * 255).byte().numpy()


def get_shirt_box(results, img_h, img_w):
    """
    Find the largest detected person and return the shirt region.
    Shirt = upper-center of the person bounding box.
    Returns (x1, y1, x2, y2) pixel coords, or None.
    """
    result = results[0]
    if result.boxes is None:
        return None

    best_box, best_area = None, 0
    for i, cls in enumerate(result.boxes.cls):
        if int(cls) == PERSON_CLASS:
            box = result.boxes.xyxy[i].cpu().numpy()
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > best_area:
                best_area, best_box = area, box

    if best_box is None:
        return None

    bx1, by1, bx2, by2 = best_box
    bw, bh = bx2 - bx1, by2 - by1
    cx = (bx1 + bx2) / 2

    x1 = max(0, int(cx - bw * PATCH_SCALE / 2))
    x2 = min(img_w - 1, int(cx + bw * PATCH_SCALE / 2))
    y1 = max(0, int(by1 + bh * SHIRT_TOP))
    y2 = min(img_h - 1, int(by1 + bh * SHIRT_BOT))

    return (x1, y1, x2, y2)


def apply_patch(img_tensor, patch, box):
    """
    Paste patch onto img_tensor at box.
    img_tensor: [1,3,H,W]  patch: [1,3,ph,pw]  box: (x1,y1,x2,y2)
    """
    x1, y1, x2, y2 = box
    ph, pw = y2 - y1, x2 - x1
    if ph <= 0 or pw <= 0:
        return img_tensor
    resized = F.interpolate(patch, size=(ph, pw), mode="bilinear", align_corners=False)
    out = img_tensor.clone()
    out[:, :, y1:y2, x1:x2] = resized
    return out


def get_raw_output(img_tensor):
    """Run YOLOv8 backbone and return raw logits (needed for backprop)."""
    resized = F.interpolate(
        img_tensor, size=(640, 640), mode="bilinear", align_corners=False
    )
    return torch_model(resized)


print("Helpers loaded.")

---

## Cell 7: EoT Transform Suite

These simulate how the patch looks in the real world:

- **Rotation** simulates the person turning or the camera angle changing
- **Scale** simulates different distances from the camera
- **Brightness/Contrast** simulates lighting changes
- **Perspective warp** simulates fabric curvature and 3D body shape (most important for shirts)
- **Motion blur** simulates the subject walking (camera motion)
- **Noise** simulates sensor/compression artifacts

Each training step averages gradients over `EOT_N` random draws from this distribution.


In [ ]:
def apply_motion_blur(tensor, max_kernel=3):
    """
    Apply motion blur in a random direction.
    tensor: [1,3,H,W] float
    """
    k = random.choice([1, 3]) if max_kernel >= 3 else 1
    if k == 1:
        return tensor
    # Horizontal motion blur kernel
    kernel = torch.zeros(k, k, device=tensor.device)
    if random.random() > 0.5:
        kernel[k // 2, :] = 1.0 / k  # horizontal
    else:
        kernel[:, k // 2] = 1.0 / k  # vertical
    kernel = kernel.view(1, 1, k, k).expand(3, 1, k, k)
    blurred = F.conv2d(tensor, kernel, padding=k // 2, groups=3)
    return blurred


def apply_perspective_warp(tensor):
    """
    Apply a subtle random perspective warp to simulate fabric curvature.
    tensor: [1,3,H,W] float
    """
    _, _, H, W = tensor.shape
    # Corner displacements (max 5% of image dimension)
    d = 0.05
    src = np.float32([[0, 0], [W, 0], [W, H], [0, H]])
    dst = np.float32(
        [
            [random.uniform(0, d * W), random.uniform(0, d * H)],
            [W - random.uniform(0, d * W), random.uniform(0, d * H)],
            [W - random.uniform(0, d * W), H - random.uniform(0, d * H)],
            [random.uniform(0, d * W), H - random.uniform(0, d * H)],
        ]
    )
    M = cv2.getPerspectiveTransform(src, dst)

    img_np = tensor_to_img(tensor)
    warped = cv2.warpPerspective(img_np, M, (W, H))
    return img_to_tensor(warped)


def random_eot_transform(img_tensor):
    """
    Apply a random combination of real-world transformations.
    img_tensor: [1,3,H,W] float in [0,1]
    Returns: transformed tensor, same shape
    """
    t = img_tensor.clone()

    # 1. Brightness
    bf = random.uniform(*BRIGHTNESS_RANGE)
    t = t * bf

    # 2. Contrast
    cf = random.uniform(*CONTRAST_RANGE)
    mean = t.mean(dim=[2, 3], keepdim=True)
    t = (t - mean) * cf + mean

    # 3. Rotation
    angle = random.uniform(*ROT_RANGE)
    t = TF.rotate(t.squeeze(0), angle).unsqueeze(0)

    # 4. Scale (zoom in/out via center crop + resize)
    scale = random.uniform(*SCALE_RANGE)
    _, _, H, W = t.shape
    new_h = int(H * scale)
    new_w = int(W * scale)
    t_scaled = F.interpolate(
        t, size=(new_h, new_w), mode="bilinear", align_corners=False
    )
    # Pad back to original size if scaled down, or center crop if scaled up
    if scale < 1.0:
        pad_h = (H - new_h) // 2
        pad_w = (W - new_w) // 2
        t = F.pad(t_scaled, (pad_w, W - new_w - pad_w, pad_h, H - new_h - pad_h))
    else:
        start_h = (new_h - H) // 2
        start_w = (new_w - W) // 2
        t = t_scaled[:, :, start_h : start_h + H, start_w : start_w + W]

    # 5. Perspective warp (simulates fabric curvature) — applied probabilistically
    if random.random() < PERSPECTIVE_PROB:
        t = apply_perspective_warp(t)

    # 6. Motion blur
    t = apply_motion_blur(t, BLUR_MAX)

    # 7. Gaussian noise
    t = t + torch.randn_like(t) * NOISE_STD

    return t.clamp(0, 1)


# Quick sanity check: apply one transform and visualize
test_tensor = img_to_tensor(orig_rgb)
test_transformed = random_eot_transform(test_tensor)
test_np = tensor_to_img(test_transformed)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(orig_rgb)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(test_np)
axes[1].set_title("EoT Transform (sample)")
axes[1].axis("off")
plt.suptitle(
    "Transform sanity check — run this cell multiple times to see different transforms"
)
plt.tight_layout()
plt.show()

---

## Cell 8: Multi-Objective Loss Function

Four components, weighted and combined:

| Component              | Weight | What it does                                             |
| ---------------------- | ------ | -------------------------------------------------------- |
| Uncertainty (entropy)  | 0.4    | Makes YOLO confused, high entropy across class scores    |
| IoU reduction          | 0.3    | Makes the predicted mask NOT match the real person shape |
| Confidence suppression | 0.2    | Directly lowers the detection confidence score           |
| Edge disruption        | 0.1    | Corrupts segmentation boundaries specifically            |

All four minimize together. The loss goes DOWN as the attack gets better.
When loss is near zero, YOLO is completely confused.


In [ ]:
def entropy_loss(raw_output):
    """
    Maximize entropy of class predictions across all anchors.
    High entropy = model is uniformly uncertain = good attack.

    raw_output: raw logits from YOLOv8 head
    Returns: scalar loss (we MINIMIZE this, which MAXIMIZES entropy)
    """
    preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output
    # Class logits: indices 4:84 (80 COCO classes after 4 bbox coords)
    class_logits = preds[:, 4:84, :]  # [1, 80, num_anchors]
    probs = torch.softmax(class_logits, dim=1)  # [1, 80, num_anchors]
    # Entropy per anchor: -sum(p * log(p))
    entropy = -(probs * (probs + 1e-8).log()).sum(dim=1)  # [1, num_anchors]
    # We want to MAXIMIZE entropy, so loss = -entropy
    return -entropy.mean()


def confidence_loss(raw_output):
    """
    Minimize mean person confidence across all anchors.
    Person class is index 0 in COCO, which sits at raw index 4.

    Returns: scalar loss (MINIMIZE = lower confidence)
    """
    preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output
    person_logits = preds[:, 4, :]  # [1, num_anchors]
    conf = torch.sigmoid(person_logits)
    return conf.mean()


def iou_loss(raw_output, shirt_box, img_shape):
    """
    Penalize overlap between the predicted segmentation mask
    and the shirt region (where the patch is).

    Intuition: if YOLO correctly segments the torso, IoU with our
    patch region is HIGH. We want to MINIMIZE this overlap.

    If raw masks aren't accessible through logits directly, we use
    a proxy: how much the bbox overlaps with the shirt region.
    """
    preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output

    # Build a target mask: 1 inside shirt_box, 0 outside
    H, W = img_shape
    x1, y1, x2, y2 = shirt_box
    target = torch.zeros(1, 1, H, W, device=DEVICE)
    target[:, :, y1:y2, x1:x2] = 1.0

    # Proxy: use the bbox coordinate predictions to estimate mask region
    # preds[:, 0:4, :] are [cx, cy, w, h] normalized bbox coords
    # We pick the highest-confidence anchor as the 'best detection'
    conf_all = torch.sigmoid(preds[:, 4, :])  # [1, num_anchors]
    best_idx = conf_all.argmax(dim=1)  # [1]

    # Get bbox of best detection (normalized 0-1)
    cx = preds[0, 0, best_idx[0]]
    cy = preds[0, 1, best_idx[0]]
    bw = preds[0, 2, best_idx[0]]
    bh = preds[0, 3, best_idx[0]]

    # Convert to pixel coords (on 640x640 input)
    scale_x, scale_y = W / 640, H / 640
    px1 = int(((cx - bw / 2) * 640 * scale_x).clamp(0, W - 1).item())
    py1 = int(((cy - bh / 2) * 640 * scale_y).clamp(0, H - 1).item())
    px2 = int(((cx + bw / 2) * 640 * scale_x).clamp(0, W - 1).item())
    py2 = int(((cy + bh / 2) * 640 * scale_y).clamp(0, H - 1).item())

    pred_mask = torch.zeros(1, 1, H, W, device=DEVICE)
    if px2 > px1 and py2 > py1:
        pred_mask[:, :, py1:py2, px1:px2] = 1.0

    # IoU between predicted region and shirt region
    intersection = (pred_mask * target).sum()
    union = pred_mask.sum() + target.sum() - intersection
    iou = intersection / (union + 1e-6)

    # We want low IoU (mask doesn't capture shirt region correctly)
    return iou


def edge_loss(raw_output):
    """
    Disrupt the sharpness of segmentation boundaries.
    We use the variance of confidence scores as a proxy:
    clean segmentation = confident anchors at edges, confused = uniform.
    Maximizing variance = making the model randomly confident in wrong places.

    Returns: scalar loss (MINIMIZE = disrupt edges)
    """
    preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output
    person_logits = preds[:, 4, :]
    conf = torch.sigmoid(person_logits)
    # High variance = confidence is spatially inconsistent = disrupted edges
    # We MINIMIZE negative variance = MAXIMIZE variance
    return -conf.var()


def combined_loss(raw_output, shirt_box, img_shape):
    """
    Multi-objective loss combining all four components.

    All four return values where LOWER = better attack.
    Weights: uncertainty=0.4, iou=0.3, confidence=0.2, edge=0.1

    Returns: scalar total loss, dict of individual components
    """
    L_unc = entropy_loss(raw_output)
    L_iou = iou_loss(raw_output, shirt_box, img_shape)
    L_conf = confidence_loss(raw_output)
    L_edge = edge_loss(raw_output)

    total = (
        W_UNCERTAINTY * L_unc + W_IOU * L_iou + W_CONFIDENCE * L_conf + W_EDGE * L_edge
    )

    components = {
        "uncertainty": L_unc.item(),
        "iou": L_iou.item(),
        "confidence": L_conf.item(),
        "edge": L_edge.item(),
        "total": total.item(),
    }
    return total, components


print("Loss functions loaded.")

---

## Cell 9: Initialize Patch & Detect Shirt Region


In [ ]:
img_h, img_w = orig_rgb.shape[:2]
shirt_box = get_shirt_box(baseline_results, img_h, img_w)

if shirt_box is None:
    print("ERROR: No person detected. Try a different image.")
else:
    x1, y1, x2, y2 = shirt_box
    patch_h, patch_w = y2 - y1, x2 - x1
    print(f"Shirt region: x=[{x1},{x2}], y=[{y1},{y2}], size={patch_w}x{patch_h}px")

    # Visualize shirt region
    vis = orig_rgb.copy()
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 3)
    cv2.putText(
        vis, "patch here", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2
    )
    plt.figure(figsize=(8, 6))
    plt.imshow(vis)
    plt.title("Detected Shirt Region — adjust SHIRT_TOP/SHIRT_BOT in config if wrong")
    plt.axis("off")
    plt.show()

    # Initialize patch as random noise in [0.4, 0.6] (neutral gray region)
    # Starting near gray is standard — gives gradient room to move in both directions
    patch_init = torch.rand(1, 3, patch_h, patch_w, device=DEVICE) * 0.2 + 0.4
    patch_init = patch_init.clamp(0, 1)
    print(f"Patch initialized. Shape: {list(patch_init.shape)}")
    print(f"Patch pixel range: [{patch_init.min():.2f}, {patch_init.max():.2f}]")

---

## Cell 10: EoT-PGD Training with Multi-Objective Loss

This is the main training loop. Each epoch:

1. Sample `EOT_N` random transforms
2. Apply each transform to the patched image
3. Run YOLO, compute 4-component loss
4. Accumulate gradients across all transforms
5. Adam optimizer step on the patch
6. Project patch back into epsilon-ball (stay within budget)

Adam is used instead of raw sign updates because it handles the
multi-component loss much better (adaptive per-parameter learning rates).


In [ ]:
def train_patch(img_rgb, patch_init, box, epsilon, lr, epochs, eot_n):
    """
    Train adversarial patch using EoT-PGD with multi-objective loss.

    Args:
        img_rgb:    clean input image HxWx3 numpy
        patch_init: starting patch [1,3,ph,pw]
        box:        (x1,y1,x2,y2) shirt region
        epsilon:    max pixel perturbation budget
        lr:         Adam learning rate
        epochs:     training epochs
        eot_n:      transforms per epoch

    Returns:
        best_patch: best patch found [1,3,ph,pw]
        history:    dict of loss component lists over epochs
    """
    img_tensor = img_to_tensor(img_rgb)
    img_shape = (img_rgb.shape[0], img_rgb.shape[1])

    # patch is the variable being optimized
    patch = patch_init.clone().requires_grad_(True)
    patch_orig = patch_init.clone().detach()  # reference for epsilon projection

    optimizer = torch.optim.Adam([patch], lr=lr)

    history = {"total": [], "uncertainty": [], "iou": [], "confidence": [], "edge": []}
    best_loss = float("inf")
    best_patch = patch.detach().clone()

    torch_model.train()  # need train mode for gradients through BN

    for epoch in range(epochs):
        optimizer.zero_grad()

        epoch_components = {
            "uncertainty": 0,
            "iou": 0,
            "confidence": 0,
            "edge": 0,
            "total": 0,
        }
        total_loss_accum = torch.tensor(0.0, device=DEVICE)

        for _ in range(eot_n):
            # Apply patch to image
            img_patched = apply_patch(img_tensor, patch, box)

            # Apply random real-world transform
            img_transformed = random_eot_transform(img_patched)

            # Forward pass (raw logits, not post-processed)
            raw_out = get_raw_output(img_transformed)

            # Multi-objective loss
            loss, components = combined_loss(raw_out, box, img_shape)
            total_loss_accum = total_loss_accum + loss

            for k in components:
                epoch_components[k] += components[k]

        # Average loss over EoT samples
        avg_loss = total_loss_accum / eot_n
        avg_loss.backward()

        # Clip patch gradients to prevent exploding updates
        torch.nn.utils.clip_grad_norm_([patch], max_norm=1.0)

        optimizer.step()

        # Project back into epsilon-ball around original patch
        with torch.no_grad():
            delta = patch.data - patch_orig
            delta = delta.clamp(-epsilon, epsilon)
            patch.data = (patch_orig + delta).clamp(0, 1)

        # Log
        for k in epoch_components:
            history[k].append(epoch_components[k] / eot_n)

        if avg_loss.item() < best_loss:
            best_loss = avg_loss.item()
            best_patch = patch.detach().clone()

        if (epoch + 1) % 10 == 0:
            ec = {k: epoch_components[k] / eot_n for k in epoch_components}
            print(
                f"Epoch [{epoch + 1:3d}/{epochs}] "
                f"total={ec['total']:+.4f} "
                f"unc={ec['uncertainty']:+.4f} "
                f"iou={ec['iou']:+.4f} "
                f"conf={ec['confidence']:+.4f} "
                f"edge={ec['edge']:+.4f}"
            )

    torch_model.eval()
    print(f"\nTraining complete. Best total loss: {best_loss:.4f}")
    return best_patch, history


print("Training function ready.")
print(
    f"Will run {NUM_EPOCHS} epochs x {EOT_N} transforms = {NUM_EPOCHS * EOT_N} forward passes."
)
print("Starting training...")

best_patch, history = train_patch(
    orig_rgb,
    patch_init,
    shirt_box,
    epsilon=EPSILON,
    lr=LR,
    epochs=NUM_EPOCHS,
    eot_n=EOT_N,
)

---

## Cell 11: Loss Curves


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

components = ["total", "uncertainty", "iou", "confidence", "edge"]
colors = ["black", "blue", "red", "orange", "purple"]
labels = [
    "Total Loss",
    "Uncertainty (entropy) — lower = more confused model",
    "IoU — lower = mask misses person",
    "Confidence — lower = model less sure person exists",
    "Edge — lower = more disrupted boundaries",
]

for i, (comp, color, label) in enumerate(zip(components, colors, labels)):
    axes[i].plot(history[comp], color=color)
    axes[i].set_title(label, fontsize=9)
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss")
    axes[i].grid(True)

axes[5].axis("off")  # hide unused 6th subplot
plt.suptitle("Multi-Objective Loss Components Over Training", fontsize=13)
plt.tight_layout()
plt.savefig("/content/loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/loss_curves.png")

---

## Cell 12: Evaluate Attack — Before vs After


In [ ]:
# Build the adversarial image using the best patch
img_tensor = img_to_tensor(orig_rgb)
img_adv = apply_patch(img_tensor, best_patch, shirt_box)
img_adv_np = tensor_to_img(img_adv)

# Run inference on both
_, adv_conf = run_inference(img_adv_np, "After EoT-PGD Attack")

# Side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(orig_rgb)
axes[0].set_title(
    f"Clean Image\nConf: {baseline_conf:.3f}",
    color="green",
    fontsize=12,
    fontweight="bold",
)
axes[0].axis("off")

axes[1].imshow(img_adv_np)
axes[1].set_title(
    f"Adversarial Image\nConf: {adv_conf:.3f}",
    color="red",
    fontsize=12,
    fontweight="bold",
)
axes[1].axis("off")

drop = baseline_conf - adv_conf
pct = drop / baseline_conf * 100 if baseline_conf > 0 else 0

plt.suptitle(f"Confidence drop: {drop:.4f} ({pct:.1f}% reduction)", fontsize=13)
plt.tight_layout()
plt.savefig("/content/comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("=" * 50)
print(f"Baseline confidence: {baseline_conf:.4f}")
print(f"Attacked confidence: {adv_conf:.4f}")
print(f"Drop: {drop:.4f} ({pct:.1f}%)")
if pct > 50:
    print("Strong attack: >50% confidence reduction")
elif pct > 20:
    print("Moderate attack: 20-50% reduction. Try increasing EPSILON or NUM_EPOCHS.")
else:
    print("Weak attack. Increase EPSILON, NUM_EPOCHS, or EOT_N and retrain.")

---

## Cell 13: Save Patch Output


In [ ]:
def save_patch(patch_tensor, path):
    p = patch_tensor.squeeze(0).permute(1, 2, 0).cpu().detach().numpy()
    p = (p * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(p).save(path)
    print(f"Saved: {path}")


# Save patch (small, can be scaled up for printing)
save_patch(best_patch, "/content/patch_eot_pgd.png")

# Save adversarial image
Image.fromarray(img_adv_np).save("/content/adversarial_image.png")
print("Saved: /content/adversarial_image.png")

# Save patch rescaled to 300x300 for print quality preview
p = best_patch.squeeze(0).permute(1, 2, 0).cpu().detach().numpy()
p = (p * 255).clip(0, 255).astype(np.uint8)
p_large = cv2.resize(p, (300, 300), interpolation=cv2.INTER_NEAREST)
Image.fromarray(p_large).save("/content/patch_300x300.png")
print("Saved: /content/patch_300x300.png (300x300 print preview)")

# Visualize the patch itself
plt.figure(figsize=(5, 5))
plt.imshow(p_large)
plt.title("Learned Adversarial Patch (300x300 upscale)")
plt.axis("off")
plt.show()

---

## Cell 14: What to Try Next

**If the attack is weak (< 20% confidence drop):**

```python
EPSILON    = 0.2      # allow stronger perturbations
NUM_EPOCHS = 100      # train longer
EOT_N      = 20       # average over more transforms
```

**If training is too slow:**

```python
EOT_N      = 5        # fewer transforms per step
NUM_EPOCHS = 30       # fewer epochs
YOLO_MODEL = 'yolov8n-seg'  # use nano if not already
```

**To plug into your video pipeline:**

- Export `best_patch` as PNG (done in Cell 13)
- Load it in Bailey's overlay code as the pattern to stamp on the shirt mask
- The patch is already in [0,1] pixel space, matching your existing compositing logic

**To attack a harder model:**

```python
YOLO_MODEL = 'yolov8x-seg'  # largest model
```

**To test transferability:**

- Train on `yolov8n-seg`, test on `yolov8x-seg` without retraining
- If confidence still drops, the patch generalizes across model sizes

**Multi-frame training (next milestone):**

```python
# Instead of one image, loop over frames from your walking videos
frames = extract_frames('walking_video.mp4', every_n=5)
for epoch in range(NUM_EPOCHS):
    for frame in frames:
        # same training loop, different frame each time
        # this makes the patch work across all walking poses
```
